# 09 — Additional benchmark evaluation: Pima + Early Stage diabetes

**External dataset-pair experiment:** Pima (client A) and Early Stage (client B). Each client retains its full local feature space, encoder, and test set. Only shared predictor weights and detached class-level latent summaries cross simulated clients. **Not a real distributed deployment or formal privacy guarantee.**

Compare **local-only**, **V3 class-conditional prototype alignment**, and **V5 class-conditional mean/covariance (CORAL-style) alignment** with fixed splits and five training seeds. The dataset target definitions, sizes and collection processes differ. Do not compare absolute scores with the original pair as though test populations were the same.

**Important:** For Early Stage, exact duplicate records are assigned to one split only, *without removing the duplicates*; this preserves sample frequency but reduces effective independent sample size. Preprocessing is fitted on training data only. Pima physiologically impossible zeros are replaced with missing values for imputation.


In [1]:
%pip install scikit-learn torch

import os, copy, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SPLIT_SEED = 42
SEEDS = [42, 52, 62, 72, 82]
LATENT_DIM = 8
ALIGN_LAMBDA = 0.10  # predefined; do not tune on the test set
print("Seeds:", SEEDS, "| alignment lambda:", ALIGN_LAMBDA)


Note: you may need to restart the kernel to use updated packages.
Seeds: [42, 52, 62, 72, 82] | alignment lambda: 0.1


In [2]:
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
PIMA_PATH = Path('../data/diabetes.csv')
EARLY_PATH = Path('../data/diabetes_data_upload.csv')
assert PIMA_PATH.exists() and EARLY_PATH.exists(), 'Put the two CSVs in the repo data/ directory.'
df1, df2 = pd.read_csv(PIMA_PATH), pd.read_csv(EARLY_PATH)
assert {'Outcome','Glucose','BMI'}.issubset(df1.columns)
assert {'class','Gender','Polyuria'}.issubset(df2.columns)
X1 = df1.drop(columns='Outcome').copy()
y1 = df1['Outcome'].astype(int)
for col in ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']:
    X1.loc[X1[col] == 0, col] = np.nan
X2 = df2.drop(columns='class').copy()
y2 = df2['class'].map({'Positive':1,'Negative':0})
assert y2.notna().all()
y2 = y2.astype(int)

# Pima: stratified 70/10/20 (train/validation/test).
X1_full,X1_test,y1_full,y1_test = train_test_split(X1,y1,test_size=.20,random_state=SPLIT_SEED,stratify=y1)
X1_train,X1_val,y1_train,y1_val = train_test_split(X1_full,y1_full,test_size=.125,random_state=SPLIT_SEED,stratify=y1_full)

# Early Stage: group by exact original row, so no duplicate row crosses splits.
# GroupShuffleSplit is group-safe but does not guarantee exact class balance.
groups=pd.util.hash_pandas_object(df2, index=False).to_numpy()
def group_split(X,y,groups,test_size,seed):
    splitter=GroupShuffleSplit(n_splits=1,test_size=test_size,random_state=seed)
    train_idx,test_idx=next(splitter.split(X,y,groups=groups))
    return train_idx,test_idx
full_idx,test_idx=group_split(X2,y2,groups,.20,SPLIT_SEED)
train_sub,val_sub=group_split(X2.iloc[full_idx],y2.iloc[full_idx],groups[full_idx],.125,SPLIT_SEED)
train_idx,val_idx=full_idx[train_sub],full_idx[val_sub]
X2_train,y2_train=X2.iloc[train_idx],y2.iloc[train_idx]
X2_val,y2_val=X2.iloc[val_idx],y2.iloc[val_idx]
X2_test,y2_test=X2.iloc[test_idx],y2.iloc[test_idx]
assert not (set(groups[train_idx]) & set(groups[val_idx]))
assert not (set(groups[train_idx]) & set(groups[test_idx]))
assert not (set(groups[val_idx]) & set(groups[test_idx]))
for name,ys in [('Pima train',y1_train),('Pima val',y1_val),('Pima test',y1_test),('Early train',y2_train),('Early val',y2_val),('Early test',y2_test)]:
    print(name, 'n=',len(ys),'positives=',int(ys.sum()))
    assert ys.nunique()==2, f'{name} has only one class; change split seed.'
print('Early Stage duplicate-safe split verified')


Pima train n= 537 positives= 187
Pima val n= 77 positives= 27
Pima test n= 154 positives= 54
Early train n= 368 positives= 220
Early val n= 45 positives= 28
Early test n= 107 positives= 72
Early Stage duplicate-safe split verified


In [3]:
def build_preprocessor(X):
    cat=X.select_dtypes(include=["object","category","bool"]).columns.tolist()
    num=[c for c in X.columns if c not in cat]
    return ColumnTransformer([
        ("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),num),
        ("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),cat)
    ])

prep1,prep2=build_preprocessor(X1_train),build_preprocessor(X2_train)
A_train=prep1.fit_transform(X1_train).astype(np.float32); A_val=prep1.transform(X1_val).astype(np.float32); A_test=prep1.transform(X1_test).astype(np.float32)
B_train=prep2.fit_transform(X2_train).astype(np.float32); B_val=prep2.transform(X2_val).astype(np.float32); B_test=prep2.transform(X2_test).astype(np.float32)
yA_train=np.asarray(y1_train,dtype=np.int64); yB_train=np.asarray(y2_train,dtype=np.int64)
print("Input dims:",A_train.shape[1],B_train.shape[1])


Input dims: 8 31


In [4]:
class ClientEncoder(nn.Module):
    def __init__(self,input_dim,latent_dim=LATENT_DIM):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(input_dim,32),nn.ReLU(),nn.Linear(32,latent_dim),nn.ReLU())
    def forward(self,x): return self.net(x)

class SharedPredictor(nn.Module):
    def __init__(self,latent_dim=LATENT_DIM):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(latent_dim,8),nn.ReLU(),nn.Linear(8,1))
    def forward(self,z): return self.net(z).squeeze(1)

class ClientModel(nn.Module):
    def __init__(self,encoder,predictor):
        super().__init__(); self.encoder=encoder; self.predictor=predictor
    def forward(self,x): return self.predictor(self.encoder(x))

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def loader(X,y,batch=512):
    return DataLoader(TensorDataset(torch.tensor(X,dtype=torch.float32),torch.tensor(np.asarray(y),dtype=torch.float32)),batch_size=batch,shuffle=True)

def train_standard(model,X,y,epochs=5,lr=1e-3):
    opt=torch.optim.Adam(model.parameters(),lr=lr); loss_fn=nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        for xb,yb in loader(X,y):
            opt.zero_grad(); loss=loss_fn(model(xb),yb); loss.backward(); opt.step()
    return model

def predict_prob(model,X):
    model.eval()
    with torch.no_grad(): return torch.sigmoid(model(torch.tensor(X,dtype=torch.float32))).cpu().numpy()

def best_threshold(y,p):
    ts=np.linspace(.05,.95,181)
    fs=[f1_score(y,(p>=t).astype(int),zero_division=0) for t in ts]
    return float(ts[int(np.argmax(fs))])

def get_metrics(y,p,t):
    pred=(p>=t).astype(int)
    return {"Threshold":t,"Accuracy":accuracy_score(y,pred),"Precision":precision_score(y,pred,zero_division=0),
            "Recall":recall_score(y,pred,zero_division=0),"F1":f1_score(y,pred,zero_division=0),"ROC-AUC":roc_auc_score(y,p)}

def average_predictors(ps,ns):
    total=float(sum(ns)); state=copy.deepcopy(ps[0].state_dict())
    for k in state:
        state[k]=torch.zeros_like(state[k])
        for p,n in zip(ps,ns): state[k]+=p.state_dict()[k]*(n/total)
    result=SharedPredictor(); result.load_state_dict(state); return result


In [5]:
# Alignment objectives retained from the original V3/V5 notebooks.
import types


In [6]:
def prototype_means(encoder,X,y):
    encoder.eval()
    with torch.no_grad():
        z=encoder(torch.as_tensor(X,dtype=torch.float32))
        yt=torch.as_tensor(np.asarray(y),dtype=torch.long)
        return {c:z[yt==c].mean(0).detach().clone() for c in (0,1)}

def prototype_coral(encoder,X,y):
    encoder.eval()
    with torch.no_grad():
        z=encoder(torch.as_tensor(X,dtype=torch.float32))
        yt=torch.as_tensor(np.asarray(y),dtype=torch.long)
        result={}
        for c in (0,1):
            zc=z[yt==c]
            if len(zc)<2: raise ValueError('Class too small for covariance')
            mu=zc.mean(0); centered=zc-mu
            cov=centered.T@centered/(len(zc)-1)
            result[c]=(mu.detach().clone(),cov.detach().clone())
        return result

def train_aligned(model,X,y,target,method,epochs=3,lr=1e-3,align_lambda=ALIGN_LAMBDA):
    opt=torch.optim.Adam(model.parameters(),lr=lr)
    bce=nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        for xb,yb in loader(X,y):
            opt.zero_grad()
            z=model.encoder(xb)
            terms=[]
            for c in (0,1):
                zc=z[yb.long()==c]
                if len(zc)==0: continue
                if method=='V3':
                    terms.append((zc.mean(0)-target[c]).square().mean())
                elif method=='V5':
                    if len(zc)<2: continue
                    mu=zc.mean(0); centered=zc-mu
                    cov=centered.T@centered/(len(zc)-1)
                    other_mu,other_cov=target[c]
                    terms.append((mu-other_mu).square().mean()+(cov-other_cov).square().mean())
                else: raise ValueError(method)
            align=torch.stack(terms).mean() if terms else z.sum()*0
            (bce(model.predictor(z),yb)+align_lambda*align).backward()
            opt.step()
    return model


In [7]:
def evaluate(seed,method,rounds=5,local_epochs=3):
    set_seed(seed)
    localA=ClientModel(ClientEncoder(A_train.shape[1]),SharedPredictor())
    localB=ClientModel(ClientEncoder(B_train.shape[1]),SharedPredictor())
    train_standard(localA,A_train,y1_train,epochs=20)
    train_standard(localB,B_train,y2_train,epochs=20)
    set_seed(seed)
    encA,encB=ClientEncoder(A_train.shape[1]),ClientEncoder(B_train.shape[1])
    global_pred=SharedPredictor()
    proto_fn=prototype_means if method=='V3' else prototype_coral
    for _ in range(rounds):
        protoA,protoB=proto_fn(encA,A_train,yA_train),proto_fn(encB,B_train,yB_train)
        modelA=ClientModel(encA,copy.deepcopy(global_pred))
        modelB=ClientModel(encB,copy.deepcopy(global_pred))
        train_aligned(modelA,A_train,yA_train,protoB,method,epochs=local_epochs)
        train_aligned(modelB,B_train,yB_train,protoA,method,epochs=local_epochs)
        encA,encB=modelA.encoder,modelB.encoder
        global_pred=average_predictors([modelA.predictor,modelB.predictor],[len(A_train),len(B_train)])
    fedA=ClientModel(encA,copy.deepcopy(global_pred))
    fedB=ClientModel(encB,copy.deepcopy(global_pred))
    cases=[('Local','Pima',localA,A_val,y1_val,A_test,y1_test),
           ('Local','Early Stage',localB,B_val,y2_val,B_test,y2_test),
           (f'Federated {method}','Pima',fedA,A_val,y1_val,A_test,y1_test),
           (f'Federated {method}','Early Stage',fedB,B_val,y2_val,B_test,y2_test)]
    rows=[]
    for approach,dataset,model,Xv,yv,Xt,yt in cases:
        t=best_threshold(yv,predict_prob(model,Xv))
        rows.append({'Seed':seed,'Approach':approach,'Dataset':dataset,**get_metrics(yt,predict_prob(model,Xt),t)})
    return rows


In [8]:
rows=[]
for method in ('V3','V5'):
    for i,seed in enumerate(SEEDS,1):
        print(f'{method}: seed {seed} ({i}/{len(SEEDS)})')
        rows.extend(evaluate(seed,method))
results=pd.DataFrame(rows)
# Local-only runs are identical for V3 and V5, so retain one set of rows.
local=results[results.Approach=='Local']
for col in ['Accuracy','Precision','Recall','F1','ROC-AUC','Threshold']:
    paired=local.pivot_table(index=['Seed','Dataset'],values=col,aggfunc=['min','max'])
    assert np.allclose(paired[('min',col)],paired[('max',col)])
results=results.drop_duplicates(subset=['Seed','Dataset','Approach']).reset_index(drop=True)
assert len(results)==30
summary=results.groupby(['Dataset','Approach'])[['Accuracy','Precision','Recall','F1','ROC-AUC','Threshold']].agg(['mean','std']).round(4)
display(summary)


V3: seed 42 (1/5)
V3: seed 52 (2/5)
V3: seed 62 (3/5)
V3: seed 72 (4/5)
V3: seed 82 (5/5)
V5: seed 42 (1/5)
V5: seed 52 (2/5)
V5: seed 62 (3/5)
V5: seed 72 (4/5)
V5: seed 82 (5/5)


Accuracy         Precision          Recall          \
                             mean     std      mean     std    mean     std   
Dataset     Approach                                                          
Early Stage Federated V3   0.7308  0.0625    0.7823  0.0566  0.8472  0.1447   
            Federated V5   0.7402  0.0566    0.7848  0.0606  0.8611  0.1211   
            Local          0.7645  0.0483    0.8707  0.0770  0.7833  0.1517   
Pima        Federated V3   0.6403  0.1373    0.5210  0.1124  0.6259  0.1091   
            Federated V5   0.6403  0.1373    0.5210  0.1124  0.6259  0.1091   
            Local          0.6896  0.0354    0.5480  0.0376  0.6296  0.1276   

                              F1         ROC-AUC         Threshold          
                            mean     std    mean     std      mean     std  
Dataset     Approach                                                        
Early Stage Federated V3  0.8050  0.0588  0.8307  0.0854     0.457  0.0670  
            Federated V5  0.8147  0.0460  0.8295  0.0867     0.457  0.0670  
            Local         0.8126  0.0563  0.8709  0.0437     0.503  0.0566  
Pima        Federated V3  0.5569  0.0828  0.7127  0.1311     0.456  0.0671  
            Federated V5  0.5569  0.0828  0.7123  0.1314     0.456  0.0671  
            Local         0.5833  0.0687  0.7443  0.0379     0.454  0.0531

In [9]:
out=Path('../models');out.mkdir(exist_ok=True)
results.to_csv(out/'additional_pair_multiseed_results.csv',index=False)
summary.to_csv(out/'additional_pair_multiseed_summary.csv')
metrics=['Accuracy','Precision','Recall','F1','ROC-AUC']
p=results.pivot(index=['Seed','Dataset'],columns='Approach',values=metrics)
deltas=[]
for (seed,dataset),row in p.iterrows():
    for method in ('V3','V5'):
        r={'Seed':seed,'Dataset':dataset,'Method':method}
        for metric in metrics:
            r[f'Delta_{metric}']=row[(metric,f'Federated {method}')]-row[(metric,'Local')]
        deltas.append(r)
pd.DataFrame(deltas).to_csv(out/'additional_pair_multiseed_deltas.csv',index=False)
print('Saved 3 CSV files to',out.resolve())


Saved 3 CSV files to /Users/zainumair12gmail.com/Desktop/COMP6002-26S2-DiaPredict/models


## Interpretation and limitations

- Five seeds vary **model training randomness only**; the train/validation/test partitions are fixed.
- Local models train for 20 epochs; federated clients train 5 × 3 local epochs. Architecture-matched but not compute-budget-matched.
- Pima uses train-only median imputation for physiologically implausible zeros. Early Stage groups identical full rows to prevent duplicate leakage, but group splitting is not stratified and group-level dependence remains.
- Prototype/covariance exchange is simulated within one Python process and is **not formally private**.
- F1 threshold is chosen on validation data and evaluated once on test. Small validation/test sets produce noisy results.
- The two datasets have different populations and possibly nonidentical diabetes definitions; this is a stress test, not proof of clinical transportability.
